[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/superconductor/superconductor.ipynb)

# The Meissner Effect

This example computes the Meissner effect: a superconducting rod, cooled through its transition in a magnetic field across it, pushes the field out of itself, and once the field is switched off it keeps none; a perfect conductor, a metal with no resistance and nothing more, freezes the field in instead and keeps it after the field is gone. Below its transition a superconductor's electrons flow as two fluids, a superfluid whose share grows as the metal cools and normal electrons that conduct with resistance. The superfluid's currents screen the field from all but a thin layer at the surface, the London penetration depth. Around a ring the same superfluid holds the flux through the ring to whole flux quanta, so that a ring cooled in a field keeps a whole number of them once the field is gone. The Stockholm lecture notes set out the [two-fluid model and the London equations](https://ekmf.fysik.su.se/Supraledning/WWW/Lectures/Lecture3.pdf).

The algebra is spacetime, `x+y+z+t-`, over a cross-section of the rod, with everything independent of z: the vector potential lies along z, `potential * mv.z`, so that the magnetic field `gradient ^ (potential * mv.z)` is a bivector in planes through z, whose lines in the cross-section are the potential's level lines, and the currents run along the rod. The superfluid's momentum together with its charge times the potential, its canonical momentum, is the spacetime gradient of one phase, so its exterior derivative vanishes, `gradient ^ (momentum + charge * potential) == 0`. That one closure is both of London's equations: its part through time and space accelerates the supercurrent by the electric field, which alone describes a perfect conductor, and its part in space fixes the supercurrent by the potential itself, which expels the field. In this cross-section a phase that does not change along z has no part along z, so the canonical momentum's part along the rod is nothing, and the supercurrent is minus the superfluid's charge squared over its mass times the potential.

In the notation of the London theory the canonical momentum reads $\hbar\nabla\varphi = m^*\mathbf v_s + e^*\mathbf A$; London's first equation is $\mathbf E = \Lambda\,\partial_t\mathbf J_s$ and his second $\lambda^2\nabla\times\nabla\times\mathbf H + \mathbf H = 0$, with $\Lambda = m/(n_s e^2)$ and $\lambda^2 = m/(\mu_0 n_s e^2)$, and a field at a flat surface decays inside as $H_0\,e^{-y/\lambda}$; around a ring the fluxoid, $\Phi + \Lambda\oint\mathbf J_s\cdot\mathrm d\mathbf l$, is a whole number of flux quanta $\Phi_0 = h/2e$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA as ga
from numga.sparse import spdiag
from examples.util.animation import show_states
from examples.electromagnetism.superconductor import render, scenarios

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
# The planes of space, which read a point's coordinates; a function's gradient on a triangle is one.
Plane = ga.gatype.from_blades("x y z")                                        # Vector+
# The points of space, each the direction from the origin to it.
Point = Plane.dual().output_gatype                                            # Antivector-
# The magnetic field across the rod, in the planes through z.
Field = ga.gatype.from_blades("xz yz")                                        # Bivector+
# The rod's direction and time's, along which nothing changes.
rod_direction = mv.z.dual()                                                   # [] Point-
time = -mv.t.dual()                                                           # [] Antivector-
# A current: a direction of flow swept through time.
Current = ga.gatype.antivector()                                              # Antivector-

## The rod

The rod's cross-section is a disk of radius one inside a disk twice as large, triangulated in rings with one ring on the rod's edge. The applied field is held on the outer rim by the potential `field * height`, a uniform field across the picture. Nothing changes along the rod or in time, so a corner's hat gradient is the plane through the facing side, the rod's direction and time, over twice the triangle's area, `(rod_direction & time & sides) / double_areas`. Lengths are in rod radii, fields in units of the applied field, and times in the time a field takes to diffuse across the normal rod.

In [ ]:
# The disk around the rod: its topology, its vertices' places in the plane z = 0, and which of them lie in the rod.
topology, places, in_rod = scenarios.rod()
vertices = mv(Plane, places).dual().field()                                   # Point-[vertices]
corners = topology.incidence[2, 0]                                            # [corners] Scalar+[faces <- vertices]
sides = topology.boundary[2, 1] * (topology.d[0] * vertices)[..., None]      # [corners] Point-[faces]
# Twice a triangle's area: two of its sides joined, swept along the rod and through time.
double_areas = ((sides[..., 0] & sides[..., 1]) & rod_direction) & time       # Scalar-[faces]
areas = double_areas.norm() / 2                                               # Scalar+[faces]
gradients = (rod_direction & time & sides) / double_areas[..., None]          # [corners] Plane+[faces]
cells = (~corners * (areas / 3)).sum(axis=-1)                                 # Scalar+[vertices]
render.draw_rod(topology, vertices, in_rod);

The figure shows the disk's triangles, the rod's shaded pale grey.

## Cooling in a field

The rod starts normal in the applied field, which runs straight through it. It is cooled through its transition to a third of the transition temperature, the superfluid's share of its electrons growing as one less the fourth power of the temperature's share, held cold in the field, and then the field is switched off. Each step balances the potential's stiffness, Ampère's law, against the currents along the rod: the normal electrons' current under the electric field, minus the potential's rate of change, and the supercurrent. The stiffness is the magnetic field of each corner's potential, `gradients ^ mv.z`, read against the vacuum's excitation of another's, `Field.dual()`. The superconductor's supercurrent responds to the potential itself, since its canonical momentum along the rod is nothing; the perfect conductor's responds only to the potential's change since its superfluid formed, the electric field's work on it, and the two run as one batch. The cell that runs the cooling takes about fifteen seconds, most of it drawing the animation.

In [ ]:
# The penetration depth; the superfluid's share and the applied field at each step; the time step and the steps between
# frames.
penetration = scenarios.penetration
superfluid, applied = scenarios.schedule()                                    # [steps], [steps]
dt, stride = scenarios.dt, scenarios.stride

In [ ]:
# The rod's vertices and each vertex's height; the potential each superfluid responds to: nothing for the
# superconductor, and for the perfect conductor the potential of the unit field it was cooled in.
inside = mv.scalar(in_rod[:, None] * 1.0).field()                             # Scalar+[vertices]
heights = mv.scalar(places[:, 1:2]).field()                                   # Scalar+[vertices]
reference = mv.scalar(np.stack([0 * places[:, 1], places[:, 1]])[..., None]).field()   # [2] Scalar+[vertices]
# The magnetic field of a potential rising by one at each corner, through the plane its gradient sweeps along z, and
# the potential's stiffness, Ampère's law: each field read against the vacuum's excitation of another, gathered from
# each triangle's corners onto the vertices they join.
fields = gradients ^ mv.z                                                     # [corners] Field+[faces]
local = (fields[..., :, None] & Field.dual()(fields[..., None, :])) * areas[..., None, None]   # [corners, corners] Scalar+[faces]
stiffness = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=-1).sum(axis=-1)   # Scalar+[vertices <- vertices]
# The normal electrons' conduction over a step, in the rod's cells; the potential is held on the outer rim and free
# inside it.
conducting = spdiag(cells * inside / dt)                                      # Scalar+[vertices <- vertices]
free = topology.reading(topology.interior[0])                                 # Scalar+[free <- vertices]

In [ ]:
def cooling() -> Iterator[tuple[Scalar, Scalar]]:
    """The potential and the supercurrent `[2] Scalar+[vertices]` every `stride` steps, of the superconductor and of
    the perfect conductor."""
    potential = applied[0] * heights + 0 * reference                          # [2] Scalar+[vertices]
    for step, (share, field) in enumerate(zip(superfluid, applied)):
        # The superfluid's charge squared over its mass: its share of the electrons over the squared penetration depth.
        londons = spdiag(cells * inside * share / penetration ** 2)           # Scalar+[vertices <- vertices]
        held = field * heights                                                # Scalar+[vertices]
        solving = (free * (conducting + stiffness + londons) * ~free) * Scalar
        right = conducting * potential + londons * reference - (conducting + stiffness + londons) * held
        potential = held + ~free * solving.solve(free * right)                # [2] Scalar+[vertices]
        if step % stride == 0:
            yield potential, -(inside * share / penetration ** 2) * (potential - reference)


# The run, drawn as it comes; the animation hands back the states it drew.
frames = show_states(cooling(), render.animate, topology, vertices, in_rod, penetration, name="superconductor",
                     duration_ms=scenarios.frame_duration_ms)

The superconductor is on the left and the perfect conductor on the right, each coloured by its supercurrent, red where it runs out of the picture and blue where into it, under the field's lines. As the superconductor cools, its currents gather at its surface and push the field's lines out to flow around it, slowed by the normal electrons' eddy currents; the perfect conductor keeps the field it was cooled in. When the field is switched off, the superconductor is left with none, while the perfect conductor holds its frozen field, whose lines close around the outside.

## The penetration depth

At the end of its hold in the field, the cold superconductor's field is read in the triangles along its upper radius, where the field runs along the surface: the field `gradients ^ (potential * mv.z)`, a bivector, its strength its norm. London's second equation makes it decay into the rod as the exponential of the depth over the penetration depth.

In [ ]:
# The superconductor's potential at the end of its hold in the field, and its field, a bivector in planes through z.
held_potential = frames[(scenarios.n_normal + scenarios.n_cooling + scenarios.n_held - 1) // stride][0][0]
field = (gradients ^ ((corners * held_potential[..., None]) * mv.z)).sum(axis=-1)   # Field+[faces]
# Each triangle's middle and its distance from the rod's axis; the triangles along the upper radius, inside the rod,
# with their depth below the surface and the field's strength there.
centroids = corners.mean(axis=0) * vertices                                   # Point-[faces]
radii = centroids.squared().square_root()                                     # Scalar+[faces]
along = ((mv.x & centroids).abs() < scenarios.radial_width) & ((mv.y & centroids) > 0) & (radii < 1)   # [faces]
depths, strengths = (1 - radii).batch()[along], field.norm().batch()[along]  # [along] Scalar+, [along] Scalar+
render.draw_penetration(depths, strengths, penetration);

The circles are the field in the triangles, in units of the applied field, against their depth in penetration depths, and the line London's exponential from the shallowest of them. Near the surface the field is two thirds stronger than the applied field, crowded around the rod, and inside it falls along the exponential by three decades; deeper still a trace of field that the eddy currents have not yet let go of remains.

## Flux in a ring

Around a superconducting ring the closure integrates to a whole number. The canonical momentum's circulation around the ring is the phase's winding times Planck's constant, and the phase must come back to itself, so the ring's fluxoid, the flux through it together with its current's circulation along it, is a whole number of flux quanta. Here the field runs along the ring's axis, `strength * mv.xy`, a bivector in the plane of the cross-section: in the ring its strength screens in by London's second equation, in the hole it stands uniform, and its level lines are the current's lines. By Ampère's law the supercurrent is the strength's gradient wedged with the field's excitation, `gradients ^ mv.xy.dual()`, and London's screening weighs its kinetic energy, the current read against the momentum the superfluid's inertia gives it, `currents & inertia(currents)`, against the field's own energy. The hole's field is fixed by the fluxoid, from the field's flux through the hole and the current's circulation along the hole's edge, which is the screening's reaction there. Lengths are in ring radii and fields in flux quanta per squared radius.

In [ ]:
# The ring around its hole: its topology, its vertices' places, which triangles lie in the ring and in the hole, which
# vertices lie in the hole or on its edge, and its rim.
ring_topology, ring_places, in_ring, hole_faces, in_hole, rim = scenarios.ring()
ring_vertices = mv(Plane, ring_places).dual().field()                         # Point-[vertices]
ring_corners = ring_topology.incidence[2, 0]                                  # [corners] Scalar+[faces <- vertices]
ring_sides = ring_topology.boundary[2, 1] * (ring_topology.d[0] * ring_vertices)[..., None]   # [corners] Point-[faces]
ring_double_areas = ((ring_sides[..., 0] & ring_sides[..., 1]) & rod_direction) & time   # Scalar-[faces]
ring_areas = ring_double_areas.norm() / 2                                     # Scalar+[faces]
ring_gradients = (rod_direction & time & ring_sides) / ring_double_areas[..., None]   # [corners] Plane+[faces]
ring_cells = (~ring_corners * (ring_areas / 3)).sum(axis=-1)                  # Scalar+[vertices]
# The superfluid's inertia, per squared penetration depth: a current read along each direction of space, in the metal's
# rest frame, into the momentum that carries it.
inertia = (stack((mv.x, mv.y, mv.z)) * (Current & stack((mv.x, mv.y, mv.z)))).sum(axis=0)   # [] Plane+ <- Current-
# The supercurrent a field's strength rising by one at each corner carries by Ampère's law, the hat's gradient wedged
# with the excitation of a unit field along the axis.
currents = ring_gradients ^ mv.xy.dual()                                      # [corners] Current-[faces]
# London's screening on the ring alone: the supercurrent's kinetic energy, each current read against the momentum
# another carries, times the squared penetration depth, with the field's own energy in the cells.
ring = mv.scalar(in_ring[:, None] * 1.0).field()                              # Scalar+[faces]
local = (currents[..., :, None] & inertia(currents[..., None, :])) * (ring * ring_areas)[..., None, None]   # [corners, corners] Scalar+[faces]
ring_kinetic = (~ring_corners[:, None] * spdiag(local) * ring_corners[None, :]).sum(axis=-1).sum(axis=-1)   # Scalar+[vertices <- vertices]
screening = penetration ** 2 * ring_kinetic + spdiag((~ring_corners * (ring * ring_areas / 3)).sum(axis=-1))   # Scalar+[vertices <- vertices]

In [ ]:
# The field is free in the ring, between the hole, whose field the fluxoid sets, and the rim, held at the applied field.
on_rim = np.isin(np.arange(ring_topology.n_vertices), rim)                    # [vertices]
ring_free = ring_topology.reading(np.flatnonzero(~(in_hole | on_rim)))        # Scalar+[free <- vertices]
hole, edge = (mv.scalar(chosen[:, None] * 1.0).field() for chosen in (in_hole, on_rim))   # Scalar+[vertices], Scalar+[vertices]
# The fields of a unit field in the hole with none at the rim, and of a unit field at the rim with none in the hole.
units = stack([hole, edge])                                                   # [2] Scalar+[vertices]
screened = units + ~ring_free * (ring_free * screening * ~ring_free * Scalar).solve(-(ring_free * (screening * units)))
# Each one's fluxoid: its flux through the hole, and its current's circulation along the hole's edge.
hole_area = (mv.scalar(hole_faces[:, None] * 1.0).field() * ring_areas).sites.sum()   # [] Scalar+
fluxoids = (hole * (screening * screened)).sites.sum() + hole_area * np.array([1.0, 0.0])   # [2] Scalar+


def ring_fields(quanta: np.ndarray, applied: np.ndarray) -> Scalar:
    """The field's strength along the axis `[...] Scalar+[vertices]` of the ring holding these fluxoids `[...]`, in
    flux quanta, in these applied fields `[...]`: the unit fields in the hole and at the rim, the hole's scaled so that
    the fluxoid comes out whole."""
    hole_field = (quanta - applied * fluxoids[1]) / fluxoids[0]               # [...] Scalar+
    return hole_field * screened[0] + applied * screened[1]


# The ring holding one, two and three quanta with no field applied.
trapped = ring_fields(scenarios.trapped_quanta, 0 * scenarios.trapped_quanta)   # [quanta] Scalar+[vertices]
render.draw_trapped(ring_topology, ring_vertices, in_hole, trapped);

The figure shows the ring holding one, two and three flux quanta, coloured by the field's strength, white for none and a deepening purple for more, under the current's lines. The field fills the hole uniformly and screens into the ring within a few penetration depths of the hole's edge, where the current crowds; it is nothing at the rim, and the whole flux through the ring is the fluxoid, less the share the current's circulation holds.

A ring cooled through its transition in a field keeps, once the field is switched off, the fluxoid it chose as it cooled: the whole number of quanta whose field costs least in that applied field, the field's departure from the applied one and the current's kinetic energy together. As the cooling field grows the kept flux climbs in steps of one quantum, as the experiments of 1961 found.

In [ ]:
# The fluxoids weighed and the fields the ring is cooled in.
candidates, cooling_fields = scenarios.candidate_quanta, scenarios.cooling_fields
fields = ring_fields(candidates[:, None], cooling_fields[None, :])           # [candidates, fields] Scalar+[vertices]
# The supercurrent in each triangle: the field strength's gradient wedged with a unit axial field's excitation.
slopes = (ring_gradients * (ring_corners * fields[..., None])).sum(axis=-1)   # [candidates, fields] Plane+[faces]
currents = slopes ^ mv.xy.dual()                                              # [candidates, fields] Current-[faces]
in_ring_areas = mv.scalar(in_ring[:, None] * 1.0).field() * ring_areas        # Scalar+[faces]
departures = fields - cooling_fields[None, :]                                 # [candidates, fields] Scalar+[vertices]
kinetic = penetration ** 2 * (in_ring_areas * (currents & inertia(currents))).sites.sum()   # [candidates, fields] Scalar+
costs = (ring_cells * departures * departures).sites.sum() + kinetic
kept = candidates[costs.argmin(axis=0)]                                       # [fields]
# The flux each ring keeps once the field is switched off.
kept_flux = (ring_fields(kept, 0 * kept) * ring_cells).sites.sum()            # [fields] Scalar+
render.draw_staircase(cooling_fields, kept_flux);

The chart shows the flux the ring keeps, in flux quanta, against the field it was cooled in: nothing, then one quantum, then two, each step coming where the cooling field's flux through the ring, its hole and the layer the field reaches into, passes a half quantum.

In [ ]:
# checks
# The field decays into the cold superconductor over the penetration depth, within a tenth: the slope of its logarithm
# over the first few depths.
shallow = (depths > 0.05) & (depths < 0.4)
decay = -1 / np.polyfit(depths[shallow].to_array(), np.log(strengths[shallow].to_array()), 1)[0]
np.testing.assert_allclose(decay, penetration, rtol=0.1)
# After the field is gone, the superconductor's core holds no field and the perfect conductor's the field it froze in:
# their potentials there, nothing and the unit field's, within two hundredths.
core = np.hypot(places[:, 0], places[:, 1]) < 0.6
last = frames[-1][0].to_array().reshape(2, -1)
assert np.abs(last[0][core]).max() < 0.02
assert np.abs(last[1][core] - places[core, 1]).max() < 0.02
# The ring traps whole flux quanta: the flux through it is each fluxoid's number of quanta within two percent, the
# current's share; cooled and let go, it keeps none, one or two.
trapped_flux = (trapped * ring_cells).sites.sum().to_array()
np.testing.assert_allclose(trapped_flux, scenarios.trapped_quanta, rtol=0.02)
assert set(np.unique(kept)) == {0.0, 1.0, 2.0}
